# Checkpoint Results

This notebook inspects the current `Challenge3.py` checkpoint layout.
It loads a Challenge 3 NSGA-II checkpoint, plots the Pareto front, runs the same checkpoint evaluation helper used by the script, and saves the Pareto plot plus morphology CSV.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from Challenge3 import evaluate_checkpoint, save_morphology_csv, save_pareto_front
from evorob.utils.filesys import get_last_checkpoint_dir

In [ ]:
# Change these to inspect different runs
RESULTS_DIR = Path("results/AntHill-v0/multi")
CHECKPOINT = "latest"  # either "latest" or a specific generation number
N_EPISODES = 8          # use a small number for quick progress checks
OUTPUT_SUBDIR = "progress_eval"

In [ ]:
assert RESULTS_DIR.exists(), f"Missing results directory: {RESULTS_DIR}"

if CHECKPOINT == "latest":
    checkpoint_root = RESULTS_DIR
    latest_checkpoint = Path(get_last_checkpoint_dir(str(RESULTS_DIR)))
else:
    latest_checkpoint = RESULTS_DIR / str(CHECKPOINT)
    checkpoint_root = latest_checkpoint

assert latest_checkpoint.exists(), f"Missing checkpoint directory: {latest_checkpoint}"

x_path = latest_checkpoint / "x.npy"
f_path = latest_checkpoint / "f.npy"
x_best_path = latest_checkpoint / "x_best.npy"

population = np.load(x_path, allow_pickle=True)
fitness = np.load(f_path, allow_pickle=True)
x_best = np.load(x_best_path, allow_pickle=True)

print(f"Checkpoint root: {checkpoint_root}")
print(f"Checkpoint folder: {latest_checkpoint}")
print(f"population shape: {population.shape}")
print(f"fitness shape: {fitness.shape}")
print(f"x_best shape: {x_best.shape}")
if fitness.ndim == 2 and fitness.shape[1] >= 2:
    print(f"obj1 range: {fitness[:, 0].min():.3f} .. {fitness[:, 0].max():.3f}")
    print(f"obj2 range: {fitness[:, 1].min():.3f} .. {fitness[:, 1].max():.3f}")

In [ ]:
if fitness.ndim == 2 and fitness.shape[1] >= 2:
    plt.figure(figsize=(6, 5))
    plt.scatter(fitness[:, 0], fitness[:, 1], s=20, alpha=0.75)
    plt.xlabel("Objective 1: reward_forward + healthy_reward")
    plt.ylabel("Objective 2: -ctrl_cost")
    plt.title(f"Pareto Front at {latest_checkpoint.name}")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print("Fitness is not multi-objective; cannot plot Pareto front.")

In [ ]:
eval_dir = latest_checkpoint / OUTPUT_SUBDIR
results = evaluate_checkpoint(
    checkpoint_dir=str(checkpoint_root),
    output_dir=str(eval_dir),
    n_episodes=N_EPISODES,
)
pareto_path = save_pareto_front(
    checkpoint_dir=str(checkpoint_root),
    output_dir=str(eval_dir),
)
morphology_path = save_morphology_csv(
    checkpoint_dir=str(checkpoint_root),
    output_dir=str(eval_dir),
)
results

In [ ]:
score_path = eval_dir / "evaluation_score.txt"
print(f"score file: {score_path}")
print(f"pareto plot: {pareto_path}")
print(f"morphology csv: {morphology_path}")

if score_path.exists():
    print(score_path.read_text())
else:
    print(f"No score file found at {score_path}")